# Economic assumptions — C$65.98 margin / C$5,500 allocated monthly cost



## Inputs and justification

- Analysis window: **28 October 2025–27 October 2026**, same **1,056 oil-producing wells**.
- **Annual constant margin: C$65.98 per oil barrel.** This is the production-weighted average of quarter-matched segment benchmarks, rounded from C$65.9761. Using a single rate changes individual well results compared with quarterly pricing.
- **Allocated remaining operating cost: C$5,500 per well-month.** Rounded from C$5,493.52: apply quarterly non-energy operating-cost benchmarks to the cohort's production, subtract C$43,815,707.165 in estimated repairs, then divide the remainder by 1,056 × 12. This is an allocation scenario, not independently observed fixed cost. The residual includes costs that may be variable, shared or unavoidable.
- **Repairs:** retain positive `field_est_total` values for Rod Truck and Workover jobs on these wells within the analysis window. Fill invalid/missing/zero estimates using the selected cohort's positive category medians (expected rod C$4,811; workover C$36,813.255). These are field estimates, not final invoices. Recalculate medians from inputs rather than hard-code them. No full-row deduplication is added, preserving the original job-count convention.
- The allocation assumes repair estimates are included within reported non-energy operating expenses. Capitalized workovers, different accounting scopes or mismatched asset coverage could invalidate that assumption. Matching the segment's total netback is partly by construction, not independent validation. Do not automatically recalibrate allocated overhead when stress-testing repair estimates.

## Primary references: Lloydminster Conventional, quarterly C$/boe

Margin before non-energy costs = sales net of blending − royalties − transportation − energy.

| Quarter | Sales | Royalties | Transport | Energy | Calculated margin | Non-energy operating cost | Source |
|---|---:|---:|---:|---:|---:|---:|---|
| Q4 2025 | 62.61 | 7.87 | 3.39 | 4.03 | 47.32 | 12.86 | [MD&A, p.22](https://www.strathconaresources.com/wp-content/uploads/2024/05/Strathcona-Managements-Discussion-and-Analysis-Q4-2025-Executed.pdf) |
| Q1 2026 | 78.60 | 10.69 | 4.07 | 4.79 | 59.05 | 15.78 | [MD&A, p.18](https://www.strathconaresources.com/wp-content/uploads/2026/05/Strathcona-Managements-Discussion-and-Analysis-Q1-2026-Executed.pdf) |
| Q2 2026 | 100.61 | 14.16 | 5.20 | 4.29 | 76.96 | 15.97 | [MD&A, p.21](https://www.strathconaresources.com/wp-content/uploads/2026/08/Strathcona-Managements-Discussion-and-Analysis-Q2-2026-Executed.pdf) |
| Q3 2026 | 96.30 | 15.35 | 3.70 | 4.01 | 73.24 | 15.95 | [MD&A, p.21](https://www.strathconaresources.com/wp-content/uploads/2026/11/Strathcona-Managements-Discussion-and-Analysis-Q3-2026-Final-Executed-Version.pdf) |

Cohort production weights (bbl): Oct28–Dec31 2025: 1,328,962.346; Q1 2026: 1,842,501.664; Q2: 1,865,081.057; Q3: 1,846,383.328; Oct1–27: 503,406.479. October 2026 uses Q3 2026 as an explicit proxy. Weighted non-energy benchmark: C$15.3567 per oil barrel. Figures are retrospective information, not inputs available before the relevant quarters ended.

The filings use sales **boe**, while this model applies segment rates to produced **oil barrels**. This is an approximation; matching production scale and field names does not prove exact asset coverage or well-specific economics. Energy may also depend on fluid lifted and idle equipment, so an oil-volume margin is a simplifying assumption.

The 3×3 grid uses chosen ±20% ranges around margin and allocated monthly cost, holding repair estimates constant. These are sensitivity scenarios, not confidence intervals or an estimated royalty response. Risk-model metrics do not change just because economic inputs change; economic tiers and monetary dispatch estimates do change. The inherited C$17,849 abandonment comparison is only an illustrative review-priority screen based on one observed cost, not a representative decommissioning liability.

## Independent arithmetic check
Using the supplied cohort and raw job CSV, the C$65.98 / C$5,500 scenario flags **162 wells** (15.34%), producing **1.88%** of oil, with a combined estimated negative contribution of **C$10.13M**. Across the chosen nine scenarios, review counts range from **125 to 218**; **125 wells** are negative in all nine. These are local arithmetic checks. Run all cells in Databricks to regenerate notebook outputs, risk tiers and charts.

## Additional comparison: original C$5,000 monthly assumption

A separate two-case comparison holds the margin at C$65.98 and each well's repair estimates unchanged, while comparing allocated monthly operating costs of C$5,000 and C$5,500. C$5,000 is the original assumption, not an observed cost. The main scorecard and economic tiers continue to use C$5,500. This comparison is separate from the nine-case +/-20% grid and its all-nine robustness count. Expected local arithmetic check: 156 review candidates at C$5,000 versus 162 at C$5,500; six additional wells enter review. The comparison is saved to `workspace.default.gold_allocated_cost_comparison` when the export cell runs.


In [5]:
daily = spark.table("workspace.default.daily_production")
fail = spark.table("workspace.default.production_failures")
jobs = spark.table("workspace.default.well_jobs")

NameError: name 'spark' is not defined

In [ ]:
print("daily:", daily.count())
print("failures:", fail.count())
print("jobs:", jobs.count())


In [ ]:
daily.select("uwi", "production_date", "flow_direction", "oil_produced_m3",
             "hours_on_prod", "primary_fluid_type", "field_operating_area").show(5)
jobs.select("uwi", "job_category", "start_date").show(5)

In [ ]:
from pyspark.sql import functions as F
START, END = "2025-10-28", "2026-10-27"
NETBACK = 65.98   # Constant annual margin benchmark, C$/oil bbl, BEFORE costs below
FIXED = 5500.0   # Allocated remaining operating cost, C$/well-month; rounded scenario
BBL_PER_M3 = 6.2898

daily = daily.withColumn("production_date", F.expr("try_cast(production_date AS DATE)"))
jobs = jobs.withColumn("start_date", F.expr("try_cast(start_date AS DATE)"))

oil = (daily
  .filter((F.col("flow_direction") == "production") & F.col("production_date").between(START, END))
  .filter(~F.coalesce(F.col("primary_fluid_type"), F.lit("")).isin("Gas", "Water Source", "Polymer"))
  .groupBy("uwi")
  .agg(F.first("field_operating_area").alias("area"), F.sum("oil_produced_m3").alias("oil_m3"))
  .filter(F.col("oil_m3") > 0)
  .withColumn("bbl", F.col("oil_m3") * BBL_PER_M3))

assert oil.count() == 1056, "Cohort changed: review source data, dates and filters before proceeding."
print("Annual window:", START, "through", END, "| oil wells:", oil.count())


In [ ]:
# Retain each eligible job's positive field estimate. These are NOT final invoices.
# Compute fallback medians only within these 1,056 wells and this annual window.
repair_jobs = (jobs
  .filter(F.col("start_date").between(START, END)
          & F.col("job_category").isin("Rod Truck", "Workover"))
  .join(oil.select("uwi"), "uwi", "inner")
  .withColumn("repair_estimate", F.expr("try_cast(field_est_total AS DOUBLE)")))
valid_estimate = (F.col("repair_estimate").isNotNull()
                  & ~F.isnan("repair_estimate")
                  & (F.col("repair_estimate") > 0)
                  & (F.col("repair_estimate") < F.lit(float("inf"))))
repair_jobs = repair_jobs.withColumn("valid_estimate", valid_estimate)
medians = (repair_jobs.filter("valid_estimate")
  .groupBy("job_category")
  .agg(F.expr("percentile(repair_estimate, 0.5)").alias("category_median")))
repair_jobs = (repair_jobs.join(medians, "job_category", "left")
  .withColumn("repair_cost", F.when(F.col("valid_estimate"), F.col("repair_estimate"))
              .otherwise(F.col("category_median"))))
assert repair_jobs.filter(F.col("repair_cost").isNull()).limit(1).count() == 0, "No valid category median for fallback."

display(repair_jobs.groupBy("job_category").agg(
    F.count("*").alias("jobs"),
    F.sum((~F.col("valid_estimate")).cast("int")).alias("estimates_filled"),
    F.first("category_median").alias("fallback_median_CAD"),
    F.sum("repair_cost").alias("estimated_repair_cost_CAD")))

j = (repair_jobs.groupBy("uwi")
  .agg(F.sum((F.col("job_category") == "Rod Truck").cast("int")).alias("rod12"),
       F.sum((F.col("job_category") == "Workover").cast("int")).alias("wo12"),
       F.sum("repair_cost").alias("repair_cost12"),
       F.sum((~F.col("valid_estimate")).cast("int")).alias("repair_estimates_filled12")))
wells = oil.join(j, "uwi", "left").fillna({"rod12": 0, "wo12": 0,
    "repair_cost12": 0.0, "repair_estimates_filled12": 0})
print("wells:", wells.count(), "| total bbl (M):", round(wells.agg(F.sum("bbl")).first()[0] / 1e6, 2))
print("rod jobs:", wells.agg(F.sum("rod12")).first()[0], "| workovers:", wells.agg(F.sum("wo12")).first()[0])
print("Total estimated repair cost C$M:", round(wells.agg(F.sum("repair_cost12")).first()[0] / 1e6, 3))


## Economic scenario: constant annual margin and allocated overhead

Use **C$65.98 per oil barrel**, **C$5,500 per well-month**, and each well's own annual repair estimates. The original C$4,000/rod and C$15,000/workover placeholders are removed. A well with no eligible repairs has zero repair cost. Full-year overhead (C$66,000) is charged to each eligible well, including partial-year producers, to preserve the original comparison method.

C$5,500 is rounded from a residual allocation of about C$5,494; it is not a measured fixed or avoidable cost. Negative contribution identifies an economic review candidate, not a proven shutdown opportunity. See the source and assumption notes at the beginning.


In [ ]:
priced = (wells
  .withColumn("margin_per_bbl_CAD", F.lit(NETBACK))
  .withColumn("allocated_cost_per_month_CAD", F.lit(FIXED))
  .withColumn("annual_allocated_cost_CAD", F.lit(12 * FIXED))
  .withColumn("net", F.col("bbl") * NETBACK - 12 * FIXED - F.col("repair_cost12")))
print("Annual margin benchmark C$/bbl:", NETBACK,
      "| allocated operating cost C$/well-month:", FIXED)


In [ ]:
losers = priced.filter(F.col("net") < 0)
n = losers.count()
loss = -(losers.agg(F.sum("net")).first()[0] or 0.0)
total_bbl = priced.agg(F.sum("bbl")).first()[0]
share = (losers.agg(F.sum("bbl")).first()[0] or 0.0) / total_bbl
print("Economic review candidates:", n, "of", priced.count(),
      "| estimated negative contribution C$M:", round(loss / 1e6, 2),
      "| share of oil %:", round(100 * share, 2))


In [3]:
priced.orderBy("net").select("uwi", "area", F.round("bbl").alias("bbl"),
    "rod12", "wo12", F.round("repair_cost12").alias("estimated_repair_cost_CAD"),
    F.round("net").alias("estimated_contribution_CAD")).show(5)


NameError: name 'priced' is not defined

In [ ]:
# Chosen sensitivity range: +/-20% around BOTH base inputs, not statistical confidence bounds.
# Hold each well's repair estimates unchanged in all nine cases.
total_bbl = wells.agg(F.sum("bbl")).first()[0]
rows = []
robust_condition = F.lit(True)
for nb in (round(NETBACK * 0.8, 2), NETBACK, round(NETBACK * 1.2, 2)):
    for fx in (FIXED * 0.8, FIXED, FIXED * 1.2):
        net = F.col("bbl") * nb - 12 * fx - F.col("repair_cost12")
        robust_condition = robust_condition & (net < 0)
        r = wells.filter(net < 0).agg(F.count("*").alias("n"),
            F.sum("bbl").alias("bbl"), F.sum(net).alias("net")).first()
        rows.append((float(nb), float(fx), int(r["n"]),
                     round(100 * (r["bbl"] or 0.0) / total_bbl, 2),
                     round(-(r["net"] or 0.0) / 1e6, 2)))
grid = spark.createDataFrame(rows, ["netback", "fixed_per_month", "wells_negative", "pct_of_oil", "annual_loss_C_M"])
robust_n = wells.filter(robust_condition).count()
grid.show()
print("Negative estimated contribution in all nine scenarios:", robust_n)


In [ ]:
# Compare the original overhead assumption with the main allocated-cost scenario.
# Margin, cohort and job-level repair estimates are identical in both cases.
COMPARISON_FIXED = 5000.0
cost_comparison_wells = (wells
    .withColumn("net_at_5000", F.col("bbl") * NETBACK - 12 * COMPARISON_FIXED - F.col("repair_cost12"))
    .withColumn("net_at_base", F.col("bbl") * NETBACK - 12 * FIXED - F.col("repair_cost12")))
comparison_rows = []
for scenario, fx, contribution_column in [
    ("Original cost assumption", COMPARISON_FIXED, "net_at_5000"),
    ("Main allocated-cost scenario", FIXED, "net_at_base"),
]:
    neg = cost_comparison_wells.filter(F.col(contribution_column) < 0)
    r = neg.agg(F.count("*").alias("n"), F.sum("bbl").alias("bbl"),
                F.sum(contribution_column).alias("net")).first()
    comparison_rows.append((scenario, float(NETBACK), float(fx), int(r["n"]),
        round(100 * (r["bbl"] or 0.0) / total_bbl, 2),
        round(-(r["net"] or 0.0) / 1e6, 2)))
allocated_cost_comparison = spark.createDataFrame(comparison_rows,
    ["scenario", "margin_per_bbl_CAD", "allocated_cost_per_month_CAD",
     "wells_for_review", "review_oil_pct", "negative_contribution_magnitude_CAD_M"])
display(allocated_cost_comparison)
additional_review = cost_comparison_wells.filter(
    (F.col("net_at_5000") >= 0) & (F.col("net_at_base") < 0)).count()
print("Additional wells requiring review at the main allocated cost:", additional_review)
print("Main scorecard and tiers still use C$", FIXED, "per well-month.", sep="")


Days the well was broken

In [ ]:
from pyspark.sql import Window

down = (daily
  .filter((F.col("flow_direction") == "production") & (F.col("production_date") >= "2021-10-01")
          & (F.coalesce(F.col("hours_on_prod"), F.lit(0)) < 12))
  .select("uwi", "production_date", "field_operating_area", "downtime_code_1"))
print("down well-days:", down.count())

In [ ]:
w = Window.partitionBy("uwi").orderBy("production_date")
down = (down.withColumn("rn", F.row_number().over(w))
            .withColumn("grp", F.expr("date_sub(production_date, rn)")))

bd = (down.groupBy("uwi", "grp")
  .agg(F.min("production_date").alias("s"), F.max("production_date").alias("e"),
       F.count("*").alias("n"), F.first("field_operating_area").alias("area"),
       F.mode("downtime_code_1").alias("cause")))
print("all breakdowns:", bd.count())

bd = bd.filter((F.col("s") >= "2022-01-01") & F.col("cause").rlike("(?i)downhole|rod|pump|tubing"))
print("equipment breakdowns since 2022:", bd.count())

In [ ]:
jb = (jobs.filter(F.col("job_category").isin("Rod Truck", "Workover"))
  .groupBy("uwi", F.col("start_date").alias("sd"))
  .agg(F.max((F.col("job_category") == "Workover").cast("int")).alias("is_wo"))
  .withColumn("jtype", F.when(F.col("is_wo") == 1, "Workover").otherwise("Rod Truck")))

m = (bd.join(jb, "uwi")
  .filter((F.col("sd") >= F.col("s")) & (F.col("sd") <= F.date_add(F.col("e"), 1)))
  .withColumn("rk", F.row_number().over(Window.partitionBy("uwi", "s").orderBy("sd")))
  .filter(F.col("rk") == 1))
print("matched to a job:", m.count())

In [ ]:
prod = (daily.filter((F.col("flow_direction") == "production") & (F.coalesce(F.col("hours_on_prod"), F.lit(0)) >= 12))
        .select("uwi", F.col("production_date").alias("d"), "oil_produced_m3"))

rate = (m.select("uwi", "s").join(prod, "uwi")
  .filter((F.col("d") >= F.date_sub(F.col("s"), 30)) & (F.col("d") < F.col("s")))
  .groupBy("uwi", "s").agg(F.avg("oil_produced_m3").alias("rate")))

In [ ]:
ep = (m.join(rate, ["uwi", "s"])
  .filter(F.col("rate") > 0)
  .withColumn("wait", F.datediff(F.col("sd"), F.col("s")))
  .withColumn("lost_bbl", F.col("rate") * F.col("wait") * 6.2898))

ep.write.mode("overwrite").saveAsTable("workspace.default.breakdowns")
ep = spark.table("workspace.default.breakdowns")

st = ep.agg(F.count("*").alias("n"), F.sum("lost_bbl").alias("bbl"), F.avg("wait").alias("mean_wait"),
            F.expr("percentile(wait, 0.5)").alias("median_wait")).first()
print("breakdowns:", st["n"], "| lost bbl (k):", round(st["bbl"] / 1e3), "| median wait:", st["median_wait"], "| mean wait:", round(st["mean_wait"], 1))

In [ ]:
import pandas as pd

pdf = ep.select("uwi", "s", "sd", "area", "jtype", "rate", "wait").toPandas()
pdf["s"] = pd.to_datetime(pdf["s"])
pdf["sd"] = pd.to_datetime(pdf["sd"])
pdf["rate"] = pdf["rate"].astype(float)
print(len(pdf))

In [ ]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def simulate_exact(df, cap=None):
    """Best possible crew order. Each real job date is one crew slot. A well can use a slot
    on or after its breakdown, and (with a cap) no later than its real job date + cap days."""
    total = 0.0
    for _, g in df.groupby(["area", "jtype"]):
        s  = g["s"].values.astype("datetime64[D]").astype(int)
        sd = g["sd"].values.astype("datetime64[D]").astype(int)
        r  = g["rate"].values
        slots = np.sort(sd)
        wait = slots[None, :] - s[:, None]
        ok = wait >= 0
        if cap is not None:
            ok &= slots[None, :] <= sd[:, None] + cap
        cost = np.where(ok, r[:, None] * wait, 1e9)
        i, j = linear_sum_assignment(cost)
        assert (cost[i, j] < 1e9).all()
        total += cost[i, j].sum()
    return total * 6.2898

act_all = (pdf["rate"] * pdf["wait"]).sum() * 6.2898
for cap in (None, 14, 7):
    e = simulate_exact(pdf, cap)
    print(cap, round(e / 1e3, 1), f"{100 * (act_all - e) / act_all:.1f}% less lost oil")

In [ ]:
def simulate(df, rule, cap=None):
    """Oil lost (bbl) while waiting, if the real crew visits are handed out by `rule`."""
    total = 0.0
    for _, g in df.groupby(["area", "jtype"]):          # crews only swap within an area and job type
        g = g.sort_values("s").reset_index(drop=True)
        start = g["s"].values.astype("datetime64[D]").astype(int)    # day the well broke down
        real  = g["sd"].values.astype("datetime64[D]").astype(int)   # day it was really fixed
        rate  = g["rate"].values                                     # oil it made before breaking
        waiting, nxt = [], 0
        for day in sorted(real):                         # each real job date is one crew visit
            while nxt < len(g) and start[nxt] <= day:    # wells broken down by this day
                waiting.append(nxt); nxt += 1
            if not waiting:
                continue
            pool = waiting
            if cap is not None:                          # deadline rule: nobody waits more than `cap`
                late = [k for k in waiting if day >= real[k] + cap]   # days longer than they really did
                pool = late if late else waiting
            if rule == "value":
                pick = max(pool, key=lambda k: (rate[k], -start[k]))  # most oil first
            else:
                pick = max(pool, key=lambda k: -start[k])             # longest waiting first
            waiting.remove(pick)
            total += rate[pick] * (day - start[pick])
    return total * 6.2898

In [ ]:
daily.createOrReplaceTempView("daily")

base = spark.sql("""
SELECT uwi, production_date AS d,
       datediff(production_date, DATE'1970-01-01') AS dn,
       oil_produced_m3 AS oil,
       greatest(water_produced_m3, 0) AS w,
       hours_on_prod AS h,
       field_operating_area AS a
FROM daily
WHERE flow_direction = 'production'
""")
base.createOrReplaceTempView("base")

def win(n):
    return f"OVER (PARTITION BY uwi ORDER BY dn RANGE BETWEEN {n-1} PRECEDING AND CURRENT ROW)"

In [ ]:
from pyspark.sql import functions as F

feat = spark.sql(f"""
SELECT uwi, d, a, oil, h,
  avg(oil) {win(7)}  AS oil7,
  avg(oil) {win(30)} AS oil30,
  avg(oil) {win(90)} AS oil90,
  avg(h)   {win(7)}  AS h7,
  avg(h)   {win(30)} AS h30,
  sum(CASE WHEN oil = 0 OR oil IS NULL THEN 1 ELSE 0 END) {win(14)} AS zero14,
  sum(w)   {win(30)} AS w30s,
  sum(oil) {win(30)} AS o30s,
  dn - min(dn) OVER (PARTITION BY uwi) AS age
FROM base
""")

feat = (feat
  .filter((F.col("d") >= "2022-01-01") & (F.col("h") >= 18) & (F.col("oil") > 0))
  .selectExpr("uwi", "d", "a", "oil7", "oil30", "oil90",
              "oil7 / nullif(oil30, 0) AS r7_30",
              "oil30 / nullif(oil90, 0) AS r30_90",
              "h7", "h30", "zero14",
              "w30s / nullif(w30s + o30s, 0) AS wc30", "age"))

print(feat.count())

In [ ]:
jobs_rw = (jobs.filter(F.col("job_category").isin("Rod Truck", "Workover"))
           .select("uwi", F.col("start_date").alias("sd")).distinct())

hist = (feat.select("uwi", "d").join(jobs_rw, "uwi", "left")
  .groupBy("uwi", "d").agg(
    F.max(F.when(F.col("sd") < F.col("d"), F.col("sd"))).alias("prev_sd"),
    F.min(F.when(F.col("sd") > F.col("d"), F.col("sd"))).alias("next_sd"),
    F.sum(F.when((F.col("sd") >= F.date_sub(F.col("d"), 90)) & (F.col("sd") < F.col("d")), 1).otherwise(0)).alias("jobs90"),
    F.sum(F.when((F.col("sd") >= F.date_sub(F.col("d"), 365)) & (F.col("sd") < F.col("d")), 1).otherwise(0)).alias("jobs365")))

panel = (feat.join(hist, ["uwi", "d"])
  .withColumn("days_since_job", F.datediff("d", "prev_sd"))
  .withColumn("y", F.when(F.col("d") <= F.lit("2026-09-27").cast("date"),
        F.when(F.col("next_sd").isNotNull() & (F.datediff("next_sd", "d") <= 30), 1).otherwise(0)))
  .drop("prev_sd", "next_sd"))

panel.write.mode("overwrite").saveAsTable("workspace.default.risk_panel")
tbl = spark.table("workspace.default.risk_panel")
print(tbl.count(), tbl.filter("y IS NOT NULL").count(), round(tbl.agg(F.avg("y")).first()[0], 3))

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score

rp = spark.table("workspace.default.risk_panel").toPandas()
rp["d"] = pd.to_datetime(rp["d"])
rp["a_code"] = rp["a"].astype("category").cat.codes
print(rp.shape)

lab   = rp[rp.y.notna()]
train = lab[lab.d < "2024-09-01"]
test  = lab[lab.d >= "2025-01-01"]
print(len(train), len(test), round(train.y.mean(), 3), round(test.y.mean(), 3))

In [ ]:
def score(name, s):
    k = int(len(test) * 0.10)
    top = np.argsort(-s)[:k]
    hit = test.y.values[top].mean()
    print(f"{name:26s} AUC {roc_auc_score(test.y, s):.3f}   top-10% hit rate {hit:.3f}   lift {hit / test.y.mean():.2f}x")

score("days since last job", -test.days_since_job.fillna(9999).values)
score("jobs in last 90 days", test.jobs90.values + 0.0)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

SENSOR = ["oil7","oil30","oil90","r7_30","r30_90","h7","h30","zero14","wc30","age","a_code"]
HIST   = ["days_since_job","jobs90","jobs365"]

def fit(cols):
    mdl = HistGradientBoostingClassifier(
        max_iter=50, learning_rate=0.05, max_leaf_nodes=48, min_samples_leaf=200,
        l2_regularization=5, early_stopping=False, random_state=0,
        categorical_features=[c == "a_code" for c in cols])
    return mdl.fit(train[cols], train.y.astype(int))

m_sensor = fit(SENSOR)
score("sensor-only model", m_sensor.predict_proba(test[SENSOR])[:, 1])

In [ ]:
FULL = SENSOR + HIST
m_full = fit(FULL)
test = test.assign(p=m_full.predict_proba(test[FULL])[:, 1])
score("full model", test.p.values)

In [ ]:
test["bucket"] = pd.cut(test.p, [0, 0.05, 0.1, 0.2, 0.3, 0.5, 1])
print(test.groupby("bucket", observed=True).agg(
    well_days=("y", "size"), avg_score=("p", "mean"), actual_job_rate=("y", "mean")).round(3))

In [4]:
all_wells = np.sort(lab.uwi.unique())
res = []
for seed in range(5):
    rng = np.random.default_rng(seed)
    held = set(rng.choice(all_wells, size=int(0.3 * len(all_wells)), replace=False))
    tr_u = lab[(lab.d < "2024-09-01") & ~lab.uwi.isin(held)]
    te_u = lab[(lab.d >= "2025-01-01") & lab.uwi.isin(held)]
    mdl = HistGradientBoostingClassifier(max_iter=50, learning_rate=0.05, max_leaf_nodes=48,
            min_samples_leaf=200, l2_regularization=5, early_stopping=False, random_state=0,
            categorical_features=[c == "a_code" for c in FULL]).fit(tr_u[FULL], tr_u.y.astype(int))
    p_u = mdl.predict_proba(te_u[FULL])[:, 1]
    top = np.argsort(-p_u)[: int(len(te_u) * 0.10)]
    hit = te_u.y.values[top].mean()
    res.append((roc_auc_score(te_u.y, p_u), hit / te_u.y.mean()))
    print(f"split {seed}: {len(held)} unseen wells, AUC {res[-1][0]:.3f}, lift {res[-1][1]:.2f}x")
print(f"5 random splits: AUC {min(a for a, _ in res):.3f} to {max(a for a, _ in res):.3f}, "
      f"lift {min(l for _, l in res):.2f}x to {max(l for _, l in res):.2f}x")

NameError: name 'np' is not defined

In [ ]:
thr = test.p.quantile(0.90)
flag = test.loc[test.p >= thr, ["uwi", "d"]]

ev = jobs_rw.toPandas().rename(columns={"sd": "job_date"})
ev["job_date"] = pd.to_datetime(ev["job_date"])
ev = ev[(ev.job_date >= "2025-02-01") & (ev.job_date <= "2026-09-27")].drop_duplicates()

hits = ev.merge(flag, on="uwi")
hits = hits[(hits.job_date - hits.d).dt.days.between(1, 30)]
caught = hits[["uwi", "job_date"]].drop_duplicates()
print(f"{len(ev)} job events; {len(caught)} had a top-10% warning in the 30 days before = {100 * len(caught) / len(ev):.1f}%")

In [ ]:
last_d = rp.d.max()
recent = rp[rp.d >= last_d - pd.Timedelta(days=14)].copy()
recent["p30"] = m_full.predict_proba(recent[FULL])[:, 1]
latest = recent.sort_values("d").groupby("uwi").tail(1)[["uwi", "d", "p30"]]
print(last_d.date(), len(latest), round(latest.p30.mean(), 3))

In [ ]:
sc = priced.toPandas().merge(latest[["uwi", "p30"]], on="uwi", how="left")
print(len(sc), sc.p30.notna().sum())

In [ ]:
ABANDON = 17849  # Illustrative screen only: ONE observed final abandonment job cost; not a median or full liability.
p80 = sc.p30.quantile(0.8)

def tier(r):
    if r.net >= 0:
        if pd.isna(r.p30):
            return "No risk score"
        return "Protect" if r.p30 >= p80 else "Maintain"
    # Illustrative review ranking only; not an abandonment recommendation or NPV.
    return "Review - high priority" if (-r.net) * 3 > ABANDON else "Review - low priority"

sc["tier"] = sc.apply(tier, axis=1)
print("risk cutoff (top 20%):", round(p80, 3))
print(sc.groupby("tier").agg(
    wells=("uwi", "count"),
    oil_pct=("bbl", lambda x: round(100 * x.sum() / sc.bbl.sum(), 1)),
    avg_risk=("p30", "mean"),
    net_C_M=("net", lambda x: round(x.sum() / 1e6, 1))).round(3))

In [ ]:
cols = ["uwi", "area", "bbl", "rod12", "wo12", "repair_cost12", "p30", "net", "tier"]
print(sc[sc.tier == "Protect"].sort_values("p30", ascending=False)[cols].head(5).round(2).to_string(index=False))
print(sc[sc.uwi == "104011604225W300"][cols].round(2).to_string(index=False))

In [ ]:
sc_spark = (spark.createDataFrame(sc)
            .withColumn("p30", F.when(F.isnan("p30"), None).otherwise(F.col("p30"))))
sc_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_well_scorecard")

grid_sp = grid if hasattr(grid, "write") else spark.createDataFrame(grid)
grid_sp.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_scenario_grid")

print(spark.table("workspace.default.gold_well_scorecard").count(),
      spark.table("workspace.default.gold_scenario_grid").count())
allocated_cost_comparison.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.gold_allocated_cost_comparison")
print("Allocated-cost comparison rows:", allocated_cost_comparison.count())


In [ ]:
# Monetary dispatch gains below use the same constant annual margin benchmark.
# They are modeled opportunity values, not guaranteed cash savings.
last = pdf[pdf["s"].between(pd.Timestamp(START), pd.Timestamp(END))]
act_all = (pdf["rate"] * pdf["wait"]).sum() * 6.2898
act_12  = (last["rate"] * last["wait"]).sum() * 6.2898

policies = [("Actual (what happened)", None, None),
            ("Highest-oil-first, no limit", "exact", None),
            ("Highest-oil-first, 14-day limit", "exact", 14),
            ("Highest-oil-first, 7-day limit", "exact", 7),
            ("First-come-first-served", "fifo", None)]
rows = []
for name, rule, cap in policies:
    if rule is None:
        lost_all, lost_12 = act_all, act_12
    elif rule == "exact":
        lost_all, lost_12 = simulate_exact(pdf, cap), simulate_exact(last, cap)
    else:
        lost_all, lost_12 = simulate(pdf, rule, cap), simulate(last, rule, cap)
    rows.append(dict(policy=name,
                     lost_kbbl=round(lost_all / 1e3, 1),
                     saving_pct=round(100 * (act_all - lost_all) / act_all, 1),
                     last12_saving_cad_m=round((act_12 - lost_12) * NETBACK / 1e6, 2)))
disp = pd.DataFrame(rows)
spark.createDataFrame(disp).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.gold_dispatch_summary")
print(disp.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

BLUE, ORANGE, GREY, INK, MUTED = "#2a78d6", "#eb6834", "#9a9992", "#0b0b0b", "#52514e"

def tidy(ax):
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.tick_params(left=False, colors=MUTED)

tt = (sc.groupby("tier").agg(wells=("uwi", "count"), bbl=("bbl", "sum"), net=("net", "sum"))
        .reindex(["Protect", "Maintain", "No risk score", "Review - high priority", "Review - low priority"]).dropna(subset=["wells"]))
tt["oil_pct"] = 100 * tt.bbl / tt.bbl.sum()
names = [f"{t}\n{int(r.wells)} wells, {r.oil_pct:.1f}% of oil" for t, r in tt.iterrows()]
vals = (tt.net / 1e6).values

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(names, vals, color=[BLUE if v >= 0 else ORANGE for v in vals], height=0.55)
ax.invert_yaxis()
for i, v in enumerate(vals):
    txt = f"C${abs(v):,.1f}M" if abs(v) < 0.05 else (f"C${v:,.1f}M" if v > 0 else f"-C${-v:,.1f}M")
    ax.text(max(v, 0) + 3, i, txt, va="center", ha="left", color=INK, fontsize=10)
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_xlim(min(vals.min() * 1.6, -25), vals.max() * 1.18)
ax.set_xlabel("Estimated contribution, last 12 months (C$ millions)", color=MUTED)
ax.set_title(f"Where the money is: {int(tt.wells['Protect'])} wells to protect, {int(tt.loc[tt.index.str.startswith('Review'), 'wells'].sum())} for economic review", loc="left", color=INK, fontsize=12)
tidy(ax); plt.tight_layout(); plt.show()

In [ ]:
gp = grid.toPandas() if hasattr(grid, "toPandas") else grid
gp = gp.sort_values(["netback", "fixed_per_month"]).reset_index(drop=True)
labels = [f"margin C${r.netback:.2f}\nallocated C${r.fixed_per_month:,.0f}/month" for r in gp.itertuples()]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.bar(labels, gp.wells_negative, color=BLUE, width=0.6)
for i, v in enumerate(gp.wells_negative):
    ax.text(i, v + gp.wells_negative.max() * 0.02, int(v), ha="center", color=INK, fontsize=10)
ax.set_ylabel("Wells with negative estimated contribution", color=MUTED)
ax.set_xlabel("Margin per bbl (before fixed and repair costs) / allocated operating cost per well-month", color=MUTED)
ax.set_title(f"{robust_n} wells have negative contribution in all nine scenarios", loc="left", color=INK, fontsize=12)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.tick_params(colors=MUTED); plt.tight_layout(); plt.show()

In [ ]:
d = disp.copy()
names = [p.replace("Highest-oil-first", "Oil-first") for p in d.policy]
cols = [GREY, BLUE, BLUE, BLUE, ORANGE]

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(names, d.lost_kbbl, color=cols, height=0.55)
ax.invert_yaxis()
for i, r in enumerate(d.itertuples()):
    note = "" if i == 0 else (f"  ({r.saving_pct:.0f}% less lost)" if r.saving_pct > 0 else f"  ({-r.saving_pct:.0f}% more lost)")
    ax.text(r.lost_kbbl + 8, i, f"{r.lost_kbbl:,.0f}k bbl{note}", va="center", color=INK, fontsize=10)
ax.set_xlim(0, d.lost_kbbl.max() * 1.55)
ax.set_xlabel("Oil lost waiting for a repair crew, 2022-2026 (thousand bbl)", color=MUTED)
ax.set_title("Sending crews to the highest-oil wells first cuts lost oil", loc="left", color=INK, fontsize=12)
tidy(ax); plt.tight_layout(); plt.show()